## Create Base Model

In [2]:
from sqlalchemy.orm import DeclarativeBase

class Base(DeclarativeBase):
    pass


### Create Models

In [3]:
from typing import Optional
from sqlalchemy import ForeignKey, String
from sqlalchemy.orm import Mapped,mapped_column,relationship



class User(Base):
    __tablename__ = "user_account"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[Optional[str]]
    addresses: Mapped[list["Address"]] = relationship(
        back_populates="user", cascade="all, delete-orphan"
    )
    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"

class Address(Base):
    __tablename__ = "address"
    id: Mapped[int] = mapped_column(primary_key=True)
    email_address: Mapped[str]
    user_id: Mapped[int] = mapped_column(ForeignKey("user_account.id"))
    user: Mapped["User"] = relationship(back_populates="addresses")
    def __repr__(self) -> str:
        return f"Address(id={self.id!r}, email_address={self.email_address!r})"


### Declare Engine


In [4]:
from sqlalchemy import create_engine
engine = create_engine("sqlite:///sqlite.db", echo=True)

### Create table

In [5]:
Base.metadata.create_all(engine)

2026-02-19 23:18:21,583 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-19 23:18:21,589 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2026-02-19 23:18:21,590 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-19 23:18:21,590 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("user_account")
2026-02-19 23:18:21,590 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-19 23:18:21,597 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2026-02-19 23:18:21,597 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-19 23:18:21,600 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("address")
2026-02-19 23:18:21,602 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-19 23:18:21,604 INFO sqlalchemy.engine.Engine 
CREATE TABLE user_account (
	id INTEGER NOT NULL, 
	name VARCHAR(30) NOT NULL, 
	fullname VARCHAR, 
	PRIMARY KEY (id)
)


2026-02-19 23:18:21,606 INFO sqlalchemy.engine.Engine [no key 0.00140s] ()
2026-02-19 23:18:21,613 INFO sqlalchemy.engine.

### Create Session

In [6]:
from sqlalchemy.orm import sessionmaker
Session = sessionmaker(bind=engine)

### Create Data

In [7]:
from sqlalchemy.orm import sessionmaker
Session = sessionmaker(bind=engine)
with Session() as session:
    spongebob = User(
        name="spongebob",
        fullname="Spongebob Squarepants",
        addresses=[Address(email_address="spongebob@sqlalchemy.org")],
    )
    sandy = User(
        name="sandy",
        fullname="Sandy Cheeks",
        addresses=[
            Address(email_address="sandy@sqlalchemy.org"),
            Address(email_address="sandy@squirrelpower.org"),
        ],
    )
    patrick = User(name="patrick", fullname="Patrick Star")
    session.add_all([spongebob, sandy, patrick])
    session.commit()

2026-02-19 23:18:21,673 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-19 23:18:21,675 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?) RETURNING id
2026-02-19 23:18:21,683 INFO sqlalchemy.engine.Engine [generated in 0.00024s (insertmanyvalues) 1/3 (ordered; batch not supported)] ('spongebob', 'Spongebob Squarepants')
2026-02-19 23:18:21,691 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?) RETURNING id
2026-02-19 23:18:21,691 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/3 (ordered; batch not supported)] ('sandy', 'Sandy Cheeks')
2026-02-19 23:18:21,691 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?) RETURNING id
2026-02-19 23:18:21,691 INFO sqlalchemy.engine.Engine [insertmanyvalues 3/3 (ordered; batch not supported)] ('patrick', 'Patrick Star')
2026-02-19 23:18:21,703 INFO sqlalchemy.engine.Engine INSERT INTO address (email_address, user_id) VALUES (?, ?) RETURN